<a href="https://colab.research.google.com/github/ys23-lys/ESAA/blob/main/ESAA_OB_WEEK5_%EC%88%98%EC%83%81%EC%9E%91%EB%A6%AC%EB%B7%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**주제** 스트레스 지수 예측

https://dacon.io/competitions/official/236526/codeshare/12854?page=1&dtype=recent

**코드 흐름**

**전처리**
- 결측값 채우기
- gender, activity, smoke_status, edu_level, sleep_pattern 범주형 변수 매핑
- medical_history, family_medical_history, smoke_status 더미변수화
- 위의 함수들을 원본 데이터프레임에 concat한 뒤 원본 칼럼 삭제.

**모델 학습**
- **전처리**: bmi 파생변수 생성 후 추가
- SVR 하이퍼 파라미터 튜닝 진행
- RobustScaler로 스케일 진행.

**새롭게 알게 된 내용 / 어려운 내용 / 배울 점**

concat을 통해 원본 데이터프레임에 변수를 추가하는 방법을 다시 한 번 습득할 수 있었다 또, SVR 하이퍼 파라미터 튜닝을 진행했는데, 서포트 벡터 머신(SVM)을 회귀에 적용한 것으로, 연속적인 수치 데이터를 예측할 때 SVR을 이용할 수 있다는 점 또한 알 수 있었다.

In [ ]:
# 더미변수화
mh_dummies    = pd.get_dummies(df['medical_history'], prefix="mh", dtype='int')
fmh_dummies   = pd.get_dummies(df['family_medical_history'], prefix="fmh", dtype='int')
smo_dummies   = pd.get_dummies(df['smoke_status'], prefix="smo", dtype='int')

In [ ]:
# bmi 파생변수 추가
df['bmi'] = (df['weight'] / ((df['height'] / 100.0) ** 2)).round(2)

In [ ]:
# SVR을 이용한 pipeline

pipe = make_pipeline(
    RobustScaler(),  # X 스케일
    TransformedTargetRegressor(
        regressor=SVR(
            **Best_params,
            kernel="rbf",
            epsilon = 0.0),
         transformer=QuantileTransformer(output_distribution="normal",
                                        n_quantiles=min(1000, len(y)))
    )
)